# Classification: Model without PCA vs Model with PCA (paired t-test)

## Imports

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, time, warnings
from scipy import stats
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_validate
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import get_scorer, classification_report, ConfusionMatrixDisplay, RocCurveDisplay
warnings.filterwarnings("ignore")

## Config

In [ ]:
DATA = "classification_data.csv"
TARGET = "Failure_Status"
ID_COL = "Record_ID"
SEED = 42
ALPHA = 0.05

## Load Data

In [ ]:
df = pd.read_csv(DATA)
df.head()

## Shape

In [ ]:
print(df.shape)

## Data Types

In [ ]:
df.info()

## Missing Values

In [ ]:
miss = df.isna().sum()
miss[miss > 0]

## Duplicate Rows

In [ ]:
print(df.duplicated().sum())

## Descriptive Statistics

In [ ]:
df.describe().T

## Identify Identifier Column

In [ ]:
df.nunique().sort_values(ascending=False).head(3)

## Feature Types

In [ ]:
X_cols = df.drop(columns=[TARGET, ID_COL])
num_cols = X_cols.select_dtypes(include="number").columns.tolist()
cat_cols = X_cols.select_dtypes(exclude="number").columns.tolist()
print("numeric:", num_cols)
print("categorical:", cat_cols)

## Class Balance

In [ ]:
print(df[TARGET].value_counts(normalize=True).round(3))
sns.countplot(x=df[TARGET])
plt.show()

## Skewness

In [ ]:
df[num_cols].skew().sort_values()

## Predictor Histograms

In [ ]:
df[num_cols].hist(bins=30, figsize=(14, 9))
plt.tight_layout()
plt.show()

## Predictor QQ Plots

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(14, 9))
for col, ax in zip(num_cols, axes.ravel()):
    stats.probplot(df[col].dropna(), plot=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## Predictors by Class (Boxplots)

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(14, 9))
for col, ax in zip(num_cols, axes.ravel()):
    sns.boxplot(x=df[TARGET], y=df[col], ax=ax)
plt.tight_layout()
plt.show()

## Categorical vs Target

In [ ]:
fig, axes = plt.subplots(1, len(cat_cols), figsize=(10, 3.5))
for col, ax in zip(cat_cols, np.atleast_1d(axes)):
    sns.countplot(x=df[col], hue=df[TARGET], ax=ax)
plt.show()

## Correlation Heatmap

In [ ]:
plt.figure(figsize=(11, 8))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.show()

## Highly Correlated Pairs

In [ ]:
corr = df[num_cols].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
pairs = upper.stack()
pairs[pairs > 0.8].sort_values(ascending=False)

## Outlier Boxplots

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(14, 8))
for col, ax in zip(num_cols, axes.ravel()):
    sns.boxplot(y=df[col], ax=ax)
plt.tight_layout()
plt.show()

## Outlier Count (IQR Rule)

In [ ]:
Q1, Q3 = df[num_cols].quantile(0.25), df[num_cols].quantile(0.75)
IQR = Q3 - Q1
outliers = (df[num_cols] < Q1 - 1.5 * IQR) | (df[num_cols] > Q3 + 1.5 * IQR)
outliers.sum()

## Drop Identifier

In [ ]:
df = df.drop(columns=ID_COL)

## Impute Categorical (fillna)

In [ ]:
df[cat_cols] = df[cat_cols].fillna("Unknown")
df[cat_cols].isna().sum()

## Encode Categorical Features

In [ ]:
df = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)
df.shape

## Encode Target

In [ ]:
le = LabelEncoder()
y = le.fit_transform(df[TARGET])
X = df.drop(columns=TARGET)
print(dict(zip(le.classes_, le.transform(le.classes_))))

## Train Test Split (stratified)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
X_train_raw, X_test_raw = X_train.copy(), X_test.copy()
print(X_train.shape, X_test.shape)

## Cross-Validation Folds (same for both models)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

## Impute Numeric (fillna with train median)

In [ ]:
med = X_train[num_cols].median()
X_train = X_train.fillna(med)
X_test = X_test.fillna(med)
print(X_train.isna().sum().sum(), X_test.isna().sum().sum())

## Outlier Treatment (IQR clip, bounds from train)

In [ ]:
Q1, Q3 = X_train[num_cols].quantile(0.25), X_train[num_cols].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
X_train[num_cols] = X_train[num_cols].clip(lower, upper, axis=1)
X_test[num_cols] = X_test[num_cols].clip(lower, upper, axis=1)

## Scale Features

In [ ]:
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

## Scoring Metrics

In [ ]:
scoring = {
    "accuracy": "accuracy",
    "precision": "precision_weighted",
    "recall": "recall_weighted",
    "f1": "f1_weighted",
    "roc_auc": "roc_auc",
}
primary = "accuracy"   # used for tuning, fold table and t-test; keep it a key of scoring

## Model Choice
Run exactly ONE of the model cells below. Run All uses the last one (SVC).

### Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=2000)
param_grid = {"C": [0.01, 0.1, 1, 10, 100]}

### KNN Classifier

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
model = KNeighborsClassifier()
param_grid = {"n_neighbors": [3, 5, 7, 9, 11, 15], "weights": ["uniform", "distance"], "p": [1, 2]}

### Decision Tree Classifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier
model = DecisionTreeClassifier(random_state=SEED)
param_grid = {"criterion": ["gini", "entropy"], "max_depth": [3, 5, 10, None], "min_samples_leaf": [1, 5, 10]}

### Random Forest Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(random_state=SEED)
param_grid = {"n_estimators": [100, 200], "max_depth": [5, 10, None], "min_samples_leaf": [1, 3, 5]}

### Gradient Boosting Classifier

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
model = GradientBoostingClassifier(random_state=SEED)
param_grid = {"n_estimators": [100, 200], "learning_rate": [0.05, 0.1], "max_depth": [2, 3, 4]}

### AdaBoost Classifier

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
model = AdaBoostClassifier(random_state=SEED)
param_grid = {"n_estimators": [50, 100, 200], "learning_rate": [0.05, 0.1, 1]}

### Gaussian Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB
model = GaussianNB()
param_grid = {"var_smoothing": [1e-9, 1e-8, 1e-7, 1e-6]}

### XGBoost Classifier

In [ ]:
from xgboost import XGBClassifier
model = XGBClassifier(random_state=SEED)
param_grid = {"n_estimators": [100, 300], "max_depth": [3, 5], "learning_rate": [0.05, 0.1]}

### SVC (RBF kernel)

In [ ]:
from sklearn.svm import SVC
model = SVC(kernel="rbf")
param_grid = {"C": [0.1, 1, 10, 100], "gamma": ["scale", 0.01, 0.1]}

## Tune Without PCA

In [ ]:
grid1 = GridSearchCV(model, param_grid, cv=cv, scoring=scoring[primary], n_jobs=-1)
grid1.fit(X_train_s, y_train)

## Best Parameters Without PCA

In [ ]:
print(grid1.best_params_)
print("CV", primary + ":", grid1.best_score_)

## Cross-Validate Without PCA

In [ ]:
res1 = cross_validate(grid1.best_estimator_, X_train_s, y_train, cv=cv, scoring=scoring,
                      return_train_score=True, n_jobs=-1)

## PCA: Components for 95% Variance

In [ ]:
pca_full = PCA().fit(X_train_s)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_comp = int(np.argmax(cumvar >= 0.95) + 1)
print("components for 95%:", n_comp)

## PCA: Explained Variance Plot

In [ ]:
plt.plot(range(1, len(cumvar) + 1), cumvar, marker="o")
plt.axhline(0.95, color="r", linestyle="--")
plt.axvline(n_comp, color="g", linestyle="--")
plt.xlabel("Components")
plt.ylabel("Cumulative explained variance")
plt.show()

## PCA: Explained Variance Table

In [ ]:
pd.DataFrame({
    "PC": range(1, n_comp + 1),
    "Explained": pca_full.explained_variance_ratio_[:n_comp],
    "Cumulative": cumvar[:n_comp],
}).round(4)

## Apply PCA

In [ ]:
pca = PCA(n_components=n_comp).fit(X_train_s)
X_train_pca = pca.transform(X_train_s)
X_test_pca = pca.transform(X_test_s)
print("original dims:", X_train_s.shape[1], "| reduced dims:", X_train_pca.shape[1])

## Tune With PCA

In [ ]:
grid2 = GridSearchCV(model, param_grid, cv=cv, scoring=scoring[primary], n_jobs=-1)
grid2.fit(X_train_pca, y_train)

## Best Parameters With PCA

In [ ]:
print(grid2.best_params_)
print("CV", primary + ":", grid2.best_score_)

## Cross-Validate With PCA

In [ ]:
res2 = cross_validate(grid2.best_estimator_, X_train_pca, y_train, cv=cv, scoring=scoring,
                      return_train_score=True, n_jobs=-1)

## Fold-wise Score Table

In [ ]:
s1 = res1[f"test_{primary}"]
s2 = res2[f"test_{primary}"]
fold_table = pd.DataFrame({
    "Fold": range(1, len(s1) + 1),
    f"{primary} Without PCA": s1,
    f"{primary} With PCA": s2,
    "Difference (With - Without)": s2 - s1,
}).round(4)
fold_table

## Mean and Std of All Metrics

In [ ]:
rows = {}
for k in scoring:
    a, b = res1[f"test_{k}"], res2[f"test_{k}"]
    rows[k.upper()] = [a.mean(), a.std(), b.mean(), b.std()]
pd.DataFrame(rows, index=["Mean (no PCA)", "Std (no PCA)", "Mean (PCA)", "Std (PCA)"]).T.round(4)

## Train vs Validation Gap

In [ ]:
gap = {}
for name, res in [("Without PCA", res1), ("With PCA", res2)]:
    tr, va = res[f"train_{primary}"].mean(), res[f"test_{primary}"].mean()
    gap[name] = {"Train": tr, "Validation": va, "Gap (Train - Val)": tr - va}
pd.DataFrame(gap).T.round(4)

## Paired t-test (two-sided)

In [ ]:
d = s2 - s1
t_stat, p_val = stats.ttest_rel(s2, s1)
p_val_main = p_val
print("mean difference:", d.mean().round(4))
print("t =", round(t_stat, 4), "| p =", round(p_val, 4))
print("Reject H0" if p_val < ALPHA else "Fail to reject H0", f"at alpha = {ALPHA}")

## Fit on Train and Evaluate on Test

In [ ]:
t0 = time.time(); m1 = grid1.best_estimator_.fit(X_train_s, y_train); fit1 = time.time() - t0
t0 = time.time(); m2 = grid2.best_estimator_.fit(X_train_pca, y_train); fit2 = time.time() - t0
test = {}
for k, s in scoring.items():
    test[k.upper()] = [get_scorer(s)(m1, X_test_s, y_test), get_scorer(s)(m2, X_test_pca, y_test)]
pd.DataFrame(test, index=["Without PCA", "With PCA"]).T.round(4)

## Confusion Matrices

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ConfusionMatrixDisplay.from_estimator(m1, X_test_s, y_test, display_labels=le.classes_, ax=ax[0], colorbar=False)
ConfusionMatrixDisplay.from_estimator(m2, X_test_pca, y_test, display_labels=le.classes_, ax=ax[1], colorbar=False)
ax[0].set_title("Without PCA"); ax[1].set_title("With PCA")
plt.show()

## Classification Reports

In [ ]:
print("Without PCA\n", classification_report(y_test, m1.predict(X_test_s), target_names=le.classes_))
print("With PCA\n", classification_report(y_test, m2.predict(X_test_pca), target_names=le.classes_))

## ROC Curves (binary)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
RocCurveDisplay.from_estimator(m1, X_test_s, y_test, name="Without PCA", ax=ax)
RocCurveDisplay.from_estimator(m2, X_test_pca, y_test, name="With PCA", ax=ax)
plt.show()

## Final Comparison Table

In [ ]:
cmp = {"Dimensions": [X_train_s.shape[1], X_train_pca.shape[1]]}
for k in scoring:
    cmp["CV " + k.upper()] = [res1[f"test_{k}"].mean(), res2[f"test_{k}"].mean()]
cmp["Train - Val gap"] = [res1[f"train_{primary}"].mean() - res1[f"test_{primary}"].mean(),
                          res2[f"train_{primary}"].mean() - res2[f"test_{primary}"].mean()]
cmp["Fit time (s)"] = [fit1, fit2]
cmp["t-test p-value"] = [p_val_main, p_val_main]
cmp["Best params"] = [str(grid1.best_params_), str(grid2.best_params_)]
pd.DataFrame(cmp, index=["Without PCA", "With PCA"]).T

## Final Answer Points (write in your own words)
- Preprocessing, dimensionality, same folds, metrics, t-test result, generalization gap, cost, interpretability, reproducibility (seed)
- Limitations: one 80/20 split, imbalanced classes, PCA components are hard to interpret, only a grid search
- Improvements: nested CV, class_weight or resampling, probability calibration, wider or randomized search

## Pipeline Version (everything inside the training folds)

In [ ]:
pipe1 = Pipeline([("imputer", SimpleImputer(strategy="median")),
                  ("scaler", StandardScaler()),
                  ("model", model)])
pipe2 = Pipeline([("imputer", SimpleImputer(strategy="median")),
                  ("scaler", StandardScaler()),
                  ("pca", PCA(n_components=0.95)),
                  ("model", model)])
pipe_grid = {"model__" + k: v for k, v in param_grid.items()}

## Pipeline Tuning Inside Folds (nested CV)

In [ ]:
inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
gs1 = GridSearchCV(pipe1, pipe_grid, cv=inner, scoring=scoring[primary], n_jobs=-1)
gs2 = GridSearchCV(pipe2, pipe_grid, cv=inner, scoring=scoring[primary], n_jobs=-1)
pres1 = cross_validate(gs1, X_train_raw, y_train, cv=cv, scoring=scoring, return_train_score=True)
pres2 = cross_validate(gs2, X_train_raw, y_train, cv=cv, scoring=scoring, return_train_score=True)

## Pipeline Fold-wise Score

In [ ]:
ps1, ps2 = pres1[f"test_{primary}"], pres2[f"test_{primary}"]
pd.DataFrame({"Fold": range(1, len(ps1) + 1), f"{primary} Without PCA": ps1,
              f"{primary} With PCA": ps2, "Difference": ps2 - ps1}).round(4)

## Pipeline Paired t-test

In [ ]:
t_stat, p_val = stats.ttest_rel(ps2, ps1)
print("t =", round(t_stat, 4), "| p =", round(p_val, 4))
print("Reject H0" if p_val < ALPHA else "Fail to reject H0", f"at alpha = {ALPHA}")

## Pipeline Final Fit and Test Score

In [ ]:
final1 = gs1.fit(X_train_raw, y_train)
final2 = gs2.fit(X_train_raw, y_train)
print(final1.best_params_, "| test", primary + ":", get_scorer(scoring[primary])(final1, X_test_raw, y_test))
print(final2.best_params_, "| test", primary + ":", get_scorer(scoring[primary])(final2, X_test_raw, y_test))